# Boltz 2b — structural binding-axis cross-check of the top clean handles (RUNG-20)

**GPU required (T4 16 GB suffices). ~10 GB Boltz weights auto-download once. ~40–60 min.**

An INDEPENDENT SOTA model (Boltz-2, AlphaFold3-class) re-asks the binding question: for each clean neoantigen handle, does the **mutant** peptide present on its MHC while the **wild-type** doesn't — so a TCR can tell self from tumour? Cross-checked 3 ways vs RUNG-11 (MHCflurry) + RUNG-12 (ESMFold).

Handles: IDH1-R132H (glioma, certified clean) ×2 alleles · BRAF-V600E (melanoma) · KRAS-G12D (the known-HARD one RUNG-12 flagged) ×2. **Set Runtime ▸ GPU.**

In [ ]:
#@title Cell 1 — clone / pull + check GPU
import os
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
!nvidia-smi -L || print('!! NO GPU — set Runtime > GPU')
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — run log + VALIDATE (selftest) + PREPARE inputs (no GPU)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG=new_log('rung20_boltz', repo_dir='.')
rc=run_logged([sys.executable,'-u','scripts/46_boltz_pmhc_specificity.py','selftest'], RUNLOG)
run_logged([sys.executable,'-u','scripts/46_boltz_pmhc_specificity.py','prepare'], RUNLOG)
print('[CELL 2]', '✓ validated + prepared' if rc==0 else f'✗ selftest FAILED ({rc}) — stop')

In [ ]:
#@title Cell 3 — install Boltz-2 (GPU; ~10 GB weights download once)
import importlib.util, sys
from scripts.runlog import run_logged
if importlib.util.find_spec('boltz') is None:
    run_logged([sys.executable,'-m','pip','install','-q','boltz'], RUNLOG, label='pip boltz')
import shutil
print('[CELL 3] boltz on PATH:', shutil.which('boltz') is not None)

In [ ]:
#@title Cell 4 — RUN: fold mut+WT pMHC for each handle, score, 3-way cross-check  [~40-60 min, resumable]
import sys, os, json
from scripts.runlog import run_logged
rc=run_logged([sys.executable,'-u','scripts/46_boltz_pmhc_specificity.py','run'], RUNLOG)
from IPython.display import Image, display
p='runs/rung20_boltz_specificity/rung20_boltz_specificity'
if os.path.exists(p+'.png'): display(Image(p+'.png'))
if os.path.exists(p+'.json'):
    d=json.load(open(p+'.json'))
    print('HEADLINE:', d['HEADLINE'])
    print('\nsummary:', json.dumps(d['summary'], indent=2))
    for r in d['results']:
        b=r.get('boltz',{}); print(f"  {r['gene']}-{r['mut_label']} {r['allele']}: D_boltz={b.get('D_boltz')} "
              f"mut_iplddt={b.get('mut_iplddt')} wt_iplddt={b.get('wt_iplddt')} -> {r['agreement']['verdict']}")
print('[CELL 4]', '✓ done' if rc==0 else f'(exit {rc}; re-run to resume)')

In [ ]:
#@title Cell 5 — bundle zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem=os.path.basename(str(RUNLOG)).replace('rung20_boltz_','').replace('.log','')
b=f'/content/rung20_boltz_run_{stem}.zip'
ps=sorted(glob.glob('runs/rung20_boltz_specificity/*.png')+glob.glob('runs/rung20_boltz_specificity/*.json')+
          glob.glob('runs/rung20_boltz_specificity/preds/*/confidence_*.json')+[str(RUNLOG)])
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=x.replace('runs/rung20_boltz_specificity/','')); print(' bundled', x)
print('[CELL 5] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(download skipped:', e, ')')